# Baltic Energy News: POS Analysis with Stanza

Scenario: analyze one news article about Baltic electricity markets to see how often different parts of speech appear. Goals: tokenize, count POS (nouns, verbs, adjectives, adverbs), compute proportions, and save a brief report.

**Files**
- Input text: `data/baltic_energy.txt`
- Output report: `output/analysis.txt`

Run all cells (outputs should be visible) before exporting to PDF.


In [1]:
"""Setup: imports, paths, and Stanza pipeline
- Downloads English tokenizer+POS models if missing
- Builds the processing pipeline
"""
from pathlib import Path
from collections import Counter
import stanza

input_path = Path("data/baltic_energy.txt")
output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / "analysis.txt"

stanza.download("en", processors="tokenize,pos", verbose=False)
nlp = stanza.Pipeline(
    "en",
    processors="tokenize,pos",
    tokenize_no_ssplit=False,
    verbose=False,
)

text = input_path.read_text(encoding="utf-8").strip()
print(f"Loaded {len(text.splitlines())} lines and {len(text.split())} raw tokens from {input_path}")



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\malle\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\malle\AppData\Local\Programs\Python\Python312\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\malle\AppData\Local\Programs\Python\Python312\Lib\site-packages\ipykernel\kernelapp.py", 

RuntimeError: Numpy is not available

In [ ]:
"""Peek at the text so we know what we're analyzing"""
preview_lines = text.splitlines()[:5]
for i, line in enumerate(preview_lines, start=1):
    print(f"{i:02d}: {line}")


In [ ]:
"""Run the pipeline and collect POS stats"""
doc = nlp(text)

sent_count = len(doc.sentences)
tokens = [w for s in doc.sentences for w in s.words]
token_count = len(tokens)

pos_counts = Counter(w.xpos for w in tokens)  # Penn Treebank
upos_counts = Counter(w.upos for w in tokens)  # Universal POS

def pos_sum(pos_list):
    return sum(pos_counts.get(tag, 0) for tag in pos_list)

n_nouns = pos_sum(["NN", "NNS", "NNP", "NNPS"])
n_verbs = pos_sum(["VB", "VBD", "VBG", "VBN", "VBP", "VBZ"])
n_adjs = pos_sum(["JJ", "JJR", "JJS"])
n_advs = pos_sum(["RB", "RBR", "RBS"])

noun_pct = n_nouns / token_count if token_count else 0
verb_pct = n_verbs / token_count if token_count else 0
adj_pct = n_adjs / token_count if token_count else 0
adv_pct = n_advs / token_count if token_count else 0

print(f"Sentences: {sent_count}")
print(f"Tokens: {token_count}")
print("-- POS counts (Penn) --")
print({"nouns": n_nouns, "verbs": n_verbs, "adjectives": n_adjs, "adverbs": n_advs})
print("-- POS proportions --")
print({
    "nouns": round(noun_pct, 3),
    "verbs": round(verb_pct, 3),
    "adjectives": round(adj_pct, 3),
    "adverbs": round(adv_pct, 3),
})


In [ ]:
"""Save a human-readable report to output/analysis.txt"""
lines = [
    f"Input file: {input_path.name}",
    f"Total sentences: {sent_count}",
    f"Total tokens: {token_count}",
    "",
    "POS counts (Penn Treebank):",
    f"  Nouns: {n_nouns}",
    f"  Verbs: {n_verbs}",
    f"  Adjectives: {n_adjs}",
    f"  Adverbs: {n_advs}",
    "",
    "POS proportions (of all tokens):",
    f"  Nouns: {noun_pct:.3f}",
    f"  Verbs: {verb_pct:.3f}",
    f"  Adjectives: {adj_pct:.3f}",
    f"  Adverbs: {adv_pct:.3f}",
    "",
    "Full POS table (Penn):",
]
lines += [f"  {tag}: {count}" for tag, count in pos_counts.most_common()]
lines += ["", "Full POS table (UPOS):"]
lines += [f"  {tag}: {count}" for tag, count in upos_counts.most_common()]

report_path.write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_path.resolve()}")
print("\nPreview:\n" + "\n".join(lines[:12]))


### Interpretation
- Higher noun share reflects factual, entity-heavy reporting (countries, energy terms).
- Verb proportion shows the text is action-oriented (imports, exports, forecasts).
- If adjectives/adverbs are relatively low, the style is informative and less opinionated.
- UPOS table can reveal overall balance across content vs. function words.
